# 25 — Deep-research verdict

Honest summary of what ML does and does not do here.

> **Reader guide.** *Aux — deep-research verdict summary:* kept for provenance. Its
> conclusions have been superseded by the current STUDY_DESIGN (Chapter 4 / Claims A + B), but
> the analysis is retained here so reviewers can trace how the earlier read arrived at the
> current retracted / retained decisions.

In [ ]:
# --- notebook preamble ---
NB_STEM = "91_deep_research_verdict"

import sys, os, json, glob
from pathlib import Path

# Make the in-repo src package importable without an install
# find repo root robustly (walks up until pyproject.toml)
_repo_root = Path.cwd()
while _repo_root != _repo_root.parent and not (_repo_root / 'pyproject.toml').is_file():
    _repo_root = _repo_root.parent
sys.path.insert(0, str(_repo_root / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from discovery9.style import apply_style, NAVY, GOLD, GREY, GREY_DASH as GREYD, CREAM, WHITE, ACTIVE, DECOY, WARN
from discovery9.paths import ROOT, RAW, DERIVED, EXTERNAL, FIGURES, TABLES, GBSA_STUDY
from discovery9.io    import load_features, load_gbsa, load_gbsa_all, load_metadata, load_bedroc_matrix, load_bedroc_all_combos, load_per_complex_analysis
from discovery9.metrics import bedroc, bedroc_per_target, rank_fuse
apply_style()

# --- fig-capture hook (iter-3 fix) ---
_SAVED_FIGS = globals().setdefault('_SAVED_FIGS', [])
_orig_figure = plt.figure
_orig_subplots = plt.subplots
def _figure_capture(*a, **kw):
    fig = _orig_figure(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig
def _subplots_capture(*a, **kw):
    fig, ax = _orig_subplots(*a, **kw)
    if fig not in _SAVED_FIGS:
        _SAVED_FIGS.append(fig)
    return fig, ax
plt.figure = _figure_capture
plt.subplots = _subplots_capture

# Legacy monolith aliases:
ACTIVE_C, DECOY_C = ACTIVE, DECOY

# Default: load the master feature table (with ligand-chem descriptors when available)
df = load_features(with_ligand_chem=True)
print(f'features.parquet: {len(df)} complexes × {df.shape[1]} columns  ·  targets: {df.target.nunique()}')


## 1. Deep-research verdict — is there an ML advantage?

We ran the full LOTO comparison on the real data (`reproduce/deep_research_wide.py`) — every model against every honest baseline. Headline: no ML model trained on MD features beats GBSA-locked, and per-target combo selection has no signal at all.

<!-- canonical baseline — see data/derived/canonical_baselines.csv -->

### Panel-mean BEDROC α=20 (higher = better)

**Convention note.** This table uses the sp-config convention from `bedroc20_partial.csv` (a different `Δ = vs locked-sp08` baseline). The `GBSA @ locked combo` row (0.609) is the 8-target-subset value for the GBSA combo `igb2_di4_salt0.15_st0.0072`. Canonical baselines (9-target panel, 4A5S imputed at 0) are docking = **0.516** [0.30, 0.73] and GBSA-locked = **0.541** [0.36, 0.71] — see `data/derived/canonical_baselines.csv`. The `Locked sp-config (sp08)` = 0.476 baseline is a genuinely different convention (best-mean sp-config on the `bedroc20_partial.csv` per-config table, not a GBSA-parameter combo).

| approach | panel BEDROC | vs locked-sp08 (Δ) |
|---|---:|---:|
| **Oracle** (per-target argmax on true BEDROC — upper bound) | **0.572** | +0.096 |
| **Locked sp-config (sp08)** — sp-config convention baseline | **0.476** | — |
| Random combo (500 draws) | 0.347 | −0.129 |
| P1 mean-baseline (predict global mean per config) | 0.476 | ±0.000 |
| P1 RidgeCV (target-fp → per-config BEDROC) | 0.349 | **−0.127** |
| P1 RandomForest | 0.298 | **−0.178** |
| P1 HGBT | 0.476 | ±0.000 |
| P2 constant classifier | 0.298 | −0.178 |
| P2 logistic (MD → is_active) | 0.445 | −0.031 |
| P2 RandomForest (MD → is_active) | **0.499** | **+0.023** |
| P2 HGBT | 0.470 | −0.006 |
| MD-composite (no fit) | 0.467 | −0.009 |
| docking_score (baseline, 8T) | 0.516 | +0.040 |
| **GBSA @ locked combo (8T, `igb2_di4_salt0.15_st0.0072`)** | **0.609** | **+0.133** |
| P3 HGBT full-factorial, ML-picks GBSA combo | 0.457 | −0.019 |
| P3 HGBT full-factorial, ML-ensemble | 0.457 | −0.019 |

### Take-home

1. **P1 (MD-fingerprint → best sp-config) has zero signal.** RidgeCV and RandomForest *hurt* vs the mean-baseline — they overfit the 9-target training set. HGBT ties the mean-baseline exactly (predicts nothing informative). With N=9, LOTO has 8 samples per fold — signal-to-noise is too low.
2. **P2 (MD → is_active) reaches docking-level performance but never touches GBSA.** RandomForest is the best per-complex ML (0.499), marginally beating locked sp-config and docking, but ~0.11 BEDROC below the 8T-GBSA (0.609) and ~0.04 below canonical 9T-GBSA (0.541). GBSA is doing something MD-stability features do not capture — most likely per-atom electrostatics and PB solvation.
3. **P3 full-factorial ML** (adds combo params + per-combo GBSA as inputs) does not improve on P2. More features, more folds, no lift. If GBSA info is available at inference, it dominates; combining it with MD features via ML buys nothing on this sample size.
4. **The temporal analysis (next cell) is the actionable finding**, not the combo-selection ML.

> **Reconciliation note.** The row *P1 RandomForest = 0.298* above comes from `reproduce/deep_research.py` (sp-config P1: per-target MD fingerprint → predict per-config BEDROC; 4-model narrow sweep). The `data/derived/deep_research_wide_summary.md` table shows `P1 RandomForest = 0.4986` — that's the wider `reproduce/deep_research_wide.py` P1 (per-target MD fingerprint → per-target BEDROC on the 9-target 4A5S-recovered panel, 7-model wide sweep, in-fold pipeline). Both are correct for their protocol; they answer different combo-selection questions on different panels. The wide-sweep P1 is what NB 30 consumes.

In [ ]:

# Load the temporal BEDROC-vs-time data from the GBSA study
# NOTE(fix-pack iter1): removed hardcoded absolute path — GBSA_STUDY comes from discovery9.paths
temporal = pd.read_csv(f'{GBSA_STUDY}/data/derived/temporal/temporal_bedroc_vs_time.csv')
frames  = pd.read_csv(f'{GBSA_STUDY}/data/derived/temporal/temporal_bedroc_vs_frames.csv')
settle  = pd.read_csv(f'{GBSA_STUDY}/data/derived/temporal/per_target_settling.csv').set_index('target')

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

# left: per-target BEDROC vs ns
ax = axes[0]
tgt_cols = [c for c in temporal.columns if c.startswith('bedroc_')]
for c in tgt_cols:
    ax.plot(temporal.ns, temporal[c], lw=1.0, color=NAVY, alpha=0.35)
ax.plot(temporal.ns, temporal.median_bedroc, lw=2.5, color=GOLD, label='median across 8 targets')
ax.set_xscale('log')
ax.set_xlabel('trajectory length [ns]  (log)')
ax.set_ylabel('BEDROC α=20')
ax.axhline(temporal.median_bedroc.iloc[-1], color=GREY, ls=':', lw=1)
ax.set_axisbelow(True); ax.grid(True, color=GREY, alpha=0.5)
ax.set_title('BEDROC α=20 vs trajectory length  ·  NAVY = per target, GOLD = median')
ax.legend()

# right: settling time histogram
ax = axes[1]
settle_sorted = settle.sort_values('settles_after_ns')
ax.barh(range(len(settle_sorted)), settle_sorted.settles_after_ns, color=NAVY, edgecolor=WHITE)
ax.set_yticks(range(len(settle_sorted)))
ax.set_yticklabels(settle_sorted.index)
ax.set_xlabel('settles after [ns]  (min. length for stable BEDROC)')
ax.set_title('Per-target BEDROC settling time')
ax.axvline(15, color=GOLD, ls='--', lw=1.5, label='15 ns')
ax.axvline(1,  color=GREY, ls=':',  lw=1)
ax.legend(loc='lower right')
ax.set_axisbelow(True); ax.xaxis.grid(True, color=GREY, alpha=0.5)
plt.tight_layout()

print('BEDROC vs frames (median across targets):')
print(frames.to_string(index=False))
print()
print('Per-target settling time (ns for BEDROC to stabilize):')
print(settle_sorted[['settles_after_ns','final']].round(3).to_string())

**Interpretation — this is the actionable finding.**

- The median across 8 targets hits its 30 ns value already at ~0.15 ns (15 frames). That's a 200× compute saving in principle.
- Per-target settling times span 0 → 20 ns. Four targets (2XU3, 5HU9, 9SI4, 9D9I) need > 12 ns; four (4L7G, 8ELC, 3I06, 4QB3) settle within 4 ns.
- So the real ML question is not "which combo per target" but "which target needs long MD?" — predict `settles_after_ns` from per-target MD features measured on a short probe run (e.g. 1 ns).

### Recommended next experiment

1. From our existing 30 ns features, extract MD signatures from just the first 1 ns (rerun with a 500-frame stride cut off at 1 ns).
2. Train a regressor: per-target 1 ns MD features → `settles_after_ns` (target from the temporal table). LOTO on 8 targets.
3. Deployment: for a new target, run 1 ns MD → predict runtime needed → run just that much.

Cleaner ML target than combo selection (single continuous label vs. discrete argmax over 20 combos) and directly saves compute — which is what the study cares about. This is where the ML advantage would show up, if there is one.

### Bottom line

- **Combo selection via ML** on this data: no signal. Don't do it.
- **MD-stability → activity classification**: reaches docking-level (0.50 BEDROC) but well below 8T-GBSA (0.609; canonical 9T-GBSA = 0.541). Not a GBSA replacement.
- **Temporal adaptivity** (predict trajectory length needed): untested but plausible — the clearest cost/benefit and the cleanest signal in the data.

In [ ]:
# --- export every figure produced in this notebook (iter-3 fix) ---
try:
    FIGURES.mkdir(parents=True, exist_ok=True)
except NameError:
    from discovery9.paths import FIGURES
    FIGURES.mkdir(parents=True, exist_ok=True)
try:
    _cream = CREAM
except NameError:
    from discovery9.style import CREAM as _cream
figs = list(globals().get('_SAVED_FIGS', []))
# fallback: any figures still open in the backend
for num in plt.get_fignums():
    f = plt.figure(num)
    if f not in figs:
        figs.append(f)
saved = []
for i, fig in enumerate(figs, start=1):
    out = FIGURES / f"{NB_STEM}_fig{i}.png"
    try:
        fig.savefig(out, bbox_inches='tight', dpi=300, facecolor=_cream)
    except Exception as e:
        print(f'  WARN: failed to save fig{i}: {e}')
        continue
    saved.append(str(out.name))
print(f'saved {len(saved)} figures:')
for s in saved:
    print(' ', s)
